In [1]:
import os
import torch
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
from scipy.stats import skew
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.utils import resample
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score,precision_score, roc_auc_score
from sklearn.decomposition import PCA
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
root_dir = 'smaller_dataset.csv'
df = pd.read_csv(root_dir)
df["Weight"] = df["Total Fwd Packet"] * df["Total Bwd packets"]

In [ ]:
categorical_columns = ["Src IP", 'Dst IP', "Src Port", "Dst Port", "Protocol"]  

for col in categorical_columns:
    df[col] = df[col].astype('category')

for col in ['Src IP', 'Dst IP', 'Src Port', 'Dst Port']:
    df[col + '_freq'] = df[col].map(df[col].value_counts())

df = pd.get_dummies(df, columns=['Protocol'], prefix='Proto')

In [4]:
df.shape

(439447, 92)

In [5]:
df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack_Type,Weight,Src IP_freq,Dst IP_freq,Src Port_freq,Dst Port_freq,Proto_0,Proto_6,Proto_17
0,192.168.137.144-192.168.137.240-49153-13217-6,192.168.137.144,49153,192.168.137.240,13217,05/08/2022 10:53:38 AM,18188538,1,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.274898,4547134.5,2.655540e+06,6473349.0,672643.0,0.0,0.0,0.0,0.0,0.0,13184858.0,4.394953e+06,3.230926e+06,6473349.0,672643.0,0,0,0,0,24,80,0.054980,0.219919,0.0,0.0,0.000000,0.000000,0.000000,0,4,1,0,1,0,0,0,4.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,1,0,29200,0,0,24,0.0,0.0,0.0,0.0,5.838632e+06,755017.935929,6473349.0,5003680.0,NeedManualLabel,DoS_DoS SYN Flood,4,1361,1363,1417,1,False,True,False
1,192.168.137.65-192.168.137.171-7661-6668-6,192.168.137.65,7661,192.168.137.171,6668,26/10/2022 11:53:56 AM,99736150,2,0,2920.0,0.0,1460.0,1460.0,1460.0,0.0,0.0,0.0,0.0,0.0,29.277248,0.020053,99736150.0,0.000000e+00,99736150.0,99736150.0,99736150.0,99736150.0,0.0,99736150.0,99736150.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0,0,0,0,40,0,0.020053,0.000000,1460.0,1460.0,1460.000000,0.000000,0.000000,0,0,0,0,2,0,0,0,0.0,2190.0,1460.0,0.0,0,0,0,0,0,0,2,2920,0,0,512,0,1,20,0.0,0.0,0.0,0.0,9.973615e+07,0.000000,99736150.0,99736150.0,NeedManualLabel,DDoS_DDoS ACK Fragmentation,0,658,116,1,9306,False,True,False
2,192.168.137.12-192.168.137.235-15376-8008-6,192.168.137.12,15376,192.168.137.235,8008,26/10/2022 03:46:03 PM,88834040,2,0,2920.0,0.0,1460.0,1460.0,1460.0,0.0,0.0,0.0,0.0,0.0,32.870283,0.022514,88834040.0,0.000000e+00,88834040.0,88834040.0,88834040.0,88834040.0,0.0,88834040.0,88834040.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0,0,0,0,40,0,0.022514,0.000000,1460.0,1460.0,1460.000000,0.000000,0.000000,0,0,0,0,2,0,0,0,0.0,2190.0,1460.0,0.0,0,0,0,0,0,0,2,2920,0,0,512,0,1,20,0.0,0.0,0.0,0.0,8.883404e+07,0.000000,88834040.0,88834040.0,NeedManualLabel,DDoS_DDoS ACK Fragmentation,0,283,777,2,960,False,True,False
3,192.168.137.12-192.168.137.206-21499-55442-6,192.168.137.12,21499,192.168.137.206,55442,26/10/2022 04:50:41 PM,17797062,1,1,1460.0,0.0,1460.0,1460.0,1460.0,0.0,0.0,0.0,0.0,0.0,82.036012,0.112378,17797062.0,0.000000e+00,17797062.0,17797062.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0,0,0,0,20,20,0.056189,0.056189,0.0,1460.0,973.333333,842.931393,710533.333333,0,0,1,0,1,0,0,0,1.0,1460.0,1460.0,0.0,0,0,0,0,0,0,1,1460,1,0,512,0,0,20,0.0,0.0,0.0,0.0,1.779706e+07,0.000000,17797062.0,17797062.0,NeedManualLabel,DDoS_DDoS ACK Fragmentation,1,283,131,1,702,False,True,False
4,192.168.137.225-192.168.137.132-38616-8080-6,192.168.137.225,38616,192.168.137.132,8080,14/09/2022 11:10:19 AM,294063,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,6.801264,294063.0,0.000000e+00,294063.0,294063.0,0

In [6]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'],  format='%d/%m/%Y %I:%M:%S %p')

In [7]:
attacks_to_remove = [
    "spoofing_ARP Spoofing",
    "spoofing_DNS Spoofing",
    "sqlinjection",
    "XSS",
    "Benign&Bruteforce_BruteForce",
    "Uploading_Attack"
]

df = df[~df["Attack_Type"].isin(attacks_to_remove)]

In [8]:
columns_to_drop = ["Flow IAT Std",
                   "Bwd Segment Size Avg",
                   "Subflow Fwd Packets",
    'Flow Duration',
    'Subflow Bwd Packets',
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Flow Packets/s',
    'Flow IAT Min',
    'Flow IAT Max',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd Header Length',
    'ACK Flag Count',
    'Packet Length Std',
    "Fwd IAT Max",
    "Idle Max",
    "Idle Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    'Average Packet Size',
    'Fwd Segment Size Avg',
    'Fwd IAT Max',
    'Bwd Header Length',
    'Packet Length Mean',
    'CWR Flag Count',
    'Average Packet Size',
    "Flow IAT Mean",
    "Active Max",
    "Bwd Bytes/Bulk Avg",
    'Fwd IAT Mean',
    'Active Mean',
    'Active Std',
    "Fwd Act Data Pkts"
]

df = df.drop(columns=columns_to_drop)
df = df.drop(columns="Label")

def create_anomaly_label(row):
    if row['Attack_Type'] == 'Benign&Bruteforce_benign':
        return 'Normal'
    else:
        return 'Attack'

df['Anomaly_Label'] = df.apply(create_anomaly_label, axis=1)

null_columns = df.isnull().sum()
null_columns = null_columns[null_columns > 0]


inf_columns = df.columns[(df == np.inf).any() | (df == -np.inf).any()]

df = df.dropna()

df = df[~df.isin([np.inf, -np.inf]).any(axis=1)]

df.replace([np.inf, -np.inf], np.nan, inplace=True)




In [ ]:
def preprocess_data(df):
    X = df.select_dtypes(include=['number', 'bool'])
    X_datetime = df.select_dtypes(include=['datetime64'])
    y = df['Anomaly_Label']  # Target variable
    y = (y == "Attack").astype(int)  # Attack = 1, Normal = 0

    
    for col in X_datetime.columns:
        X_datetime[col] = pd.to_datetime(X_datetime[col]).astype('int64') // 10**9


    scaler_numerical = StandardScaler()
    X_scaled_num = scaler_numerical.fit_transform(X)

 
    scaler_datetime = StandardScaler()
    X_scaled_datetime = scaler_datetime.fit_transform(X_datetime)


    X_scaled= np.concatenate([X_scaled_num, X_scaled_datetime], axis=1)
    

    pca = PCA(n_components=0.95)  # Retain 95% of variance

    X_scaled = pca.fit_transform(X_scaled)

    
    return X_scaled, y

In [ ]:
import csv
from datetime import datetime
import os
class CSVCallback:
    def __init__(self, filename):
        self.filename = filename
        self.file_exists = False
    
    def __call__(self, study, trial):
       
        params = trial.params
        metrics = {k: v for k, v in trial.user_attrs.items()}
        
        
        row = {
            'trial_number': trial.number,
            'f1_score': 1.0 - trial.value,  # Convert back to F1 score
            'datetime': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'state': str(trial.state),
            **params,
            **metrics
        }
        
        # Write to CSV
        header = not self.file_exists
        mode = 'w' if header else 'a'
        
        with open(self.filename, mode, newline='') as f:
            writer = csv.DictWriter(f, fieldnames=row.keys())
            if header:
                writer.writeheader()
                self.file_exists = True
            writer.writerow(row)

In [ ]:
import torch
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, activation=nn.ReLU(), dropout_rate=0.0, 
                 sequence_length=1, num_layers=1, bidirectional=False):
        super(LSTMAutoencoder, self).__init__()

        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.bottleneck_dim = hidden_dims[-1]
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.directions = 2 if bidirectional else 1
        self.activation = activation

 
        self.encoder_layers = nn.ModuleList()
        prev_dim = input_dim
        for h_dim in hidden_dims:
            self.encoder_layers.append(nn.LSTM(
                input_size=prev_dim,
                hidden_size=h_dim,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            ))
            prev_dim = h_dim * self.directions

      
        self.decoder_layers = nn.ModuleList()
        reversed_dims = list(reversed(hidden_dims))
        prev_dim = reversed_dims[0] * self.directions  # Start with bottleneck dimension


        for i in range(len(reversed_dims) - 1):
            next_dim = reversed_dims[i+1]
            self.decoder_layers.append(nn.LSTM(
                input_size=prev_dim,
                hidden_size=next_dim,
                num_layers=1,
                batch_first=True,
                bidirectional=bidirectional
            ))
            prev_dim = next_dim * self.directions

        # Final output layer    self.output_layer = nn.Linear(hidden_dims[0] * self.directions, input_dim)

        # Dropout layer
        self.dropout = nn.Dropout(dropout_rate) if dropout_rate > 0 else None

    def forward(self, x):
       
        original_shape = x.shape
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # [batch_size, 1, input_dim]

        batch_size = x.size(0)
        seq_len = x.size(1)

        # Encode
        current_input = x
        for i, encoder in enumerate(self.encoder_layers):
            outputs, (hidden, cell) = encoder(current_input)
            current_input = outputs
            
            # Apply activation
            if self.activation is not None:
                current_input = self.activation(current_input)

     
            if self.dropout is not None and i < len(self.encoder_layers) - 1:
                current_input = self.dropout(current_input)

       
        encoded = outputs[:, -1, :]

      
        decoder_input = encoded.unsqueeze(1).repeat(1, seq_len, 1)

        # Decode
        current_input = decoder_input
        for i, decoder in enumerate(self.decoder_layers):
            outputs, _ = decoder(current_input)
            current_input = outputs
            
    
            if self.activation is not None and i < len(self.decoder_layers) - 1:
                current_input = self.activation(current_input)

         
            if self.dropout is not None and i < len(self.decoder_layers) - 1:
                current_input = self.dropout(current_input)

   
        reconstructed = self.output_layer(current_input)

       
        if len(original_shape) == 2:
            reconstructed = reconstructed.squeeze(1)

        return reconstructed

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error
import numpy as np

def objective(trial):
 
    architecture = [
        [24,12],
        [20, 10],
        [16,8]
    ]
    arch_idx = trial.suggest_int("architecture_index", 0, len(architecture) - 1)
    hidden_dims = architecture[arch_idx]

    batch_size = trial.suggest_categorical("batch_size", [32])
    lr = trial.suggest_categorical("learning_rate", [1e-3])
    num_epochs = trial.suggest_categorical("epochs", [30,70])
    dropout_rate = trial.suggest_categorical("dropout_rate", [0.0, 0.1])
    activation_name = trial.suggest_categorical("activation", ["ReLU", "Tanh", "LeakyReLU"])

    if activation_name == "LeakyReLU":
        activation_fn = nn.LeakyReLU()
    else:
        activation_fn = getattr(nn, activation_name)()

    sequence_length = trial.suggest_categorical("sequence_length", [5, 10])
    bidirectional = trial.suggest_categorical("bidirectional", [False, True])
    num_layers = trial.suggest_categorical("num_layers", [1,3])

  
    trial.set_user_attr("architecture_details", str(hidden_dims))
    

    X_scaled, y = preprocess_data(df)

  
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42, stratify=y)
    X_train_full, X_val, y_train_full, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.143, random_state=42, stratify=y_train_val)
    X_train = X_train_full[y_train_full == 0]
    input_dim = X_train.shape[1]

  
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val = y_val.values if hasattr(y_val, 'values') else y_val


    train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = LSTMAutoencoder(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        activation=activation_fn,
        dropout_rate=dropout_rate,
        sequence_length=sequence_length,
        bidirectional=bidirectional,
        num_layers=num_layers
    ).to(device)

    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "RMSprop"])
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=lr)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=lr)

    criterion = nn.MSELoss()

    best_f1 = 0
    patience = 10
    wait = 0
    epoch_results = []

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience)

  
    for epoch in range(num_epochs):
        model.train()
        train_losses = []

        for batch in train_loader:
            x_batch = batch[0].to(device)
            optimizer.zero_grad()
            output = model(x_batch)
            loss = criterion(output, x_batch)
            loss.backward()

   
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_losses.append(loss.item())

        avg_train_loss = sum(train_losses) / len(train_losses)

      
        model.eval()
        with torch.no_grad():
            val_output = model(X_val_tensor.to(device))
            val_loss = criterion(val_output, X_val_tensor.to(device)).item()


            val_errors = torch.mean((val_output - X_val_tensor.to(device)) ** 2, dim=1).cpu().numpy()


            best_f1 = 0
            best_thresh = 0
            for thresh in np.linspace( 
                                        np.percentile(val_errors, 50), 
                                        np.percentile(val_errors, 99),   
                                        100):
                y_pred_temp = (val_errors > thresh).astype(int)
                f1_temp = f1_score(y_val, y_pred_temp)
                if f1_temp > best_f1:
                    best_f1 = f1_temp
                    best_thresh = thresh

          
            threshold = best_thresh
            y_pred = (val_errors > threshold).astype(int)

            scheduler.step(val_loss)

        
            f1 = best_f1
            precision = precision_score(y_val, y_pred, zero_division=0)
            recall = recall_score(y_val, y_pred, zero_division=0)
            
        
            
            # Calculate confusion matrix components
            cm = confusion_matrix(y_val, y_pred)
            tn, fp, fn, tp = cm.ravel()
            
          
            epoch_result = {
                'epoch': epoch + 1,
                'train_loss': avg_train_loss,
                'val_loss': val_loss,
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'threshold': threshold,
                'true_negatives': int(tn),
                'false_positives': int(fp),
                'false_negatives': int(fn),
                'true_positives': int(tp)
            }
            epoch_results.append(epoch_result)
            
            
            for key, value in epoch_result.items():
                trial.set_user_attr(f"last_epoch_{key}", value)

            if f1 > best_f1:
                best_f1 = f1
                wait = 0
                
               
                for key, value in epoch_result.items():
                    trial.set_user_attr(f"best_{key}", value)
            else:
                wait += 1

        if wait >= patience:
            trial.set_user_attr("early_stopping_epoch", epoch)
            break
    
  
    trial.set_user_attr("all_epochs_summary", str(epoch_results))

    return 1.0 - best_f1  

c:\Users\MELİSA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def run_optuna_study(n_trials=20, study_name="lstm_autoencoder_study"):
  
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"optuna_trials_{timestamp}.csv"
    

    csv_callback = CSVCallback(csv_filename)
    
    
    study = optuna.create_study(
        study_name=study_name,
        direction="minimize"
    )
    
 
    study.optimize(objective, n_trials=n_trials, callbacks=[csv_callback])
    
  
    best_trial = study.best_trial
    print(f"Best trial: {best_trial.number}")
    print(f"Best F1 Score: {1.0 - best_trial.value:.4f}")
    print("Best hyperparameters:")
    for key, value in best_trial.params.items():
        print(f"    {key}: {value}")
    
    
    best_results = []
    for trial in study.trials:
        if trial.state == optuna.trial.TrialState.COMPLETE:
            result = {
                'trial_number': trial.number,
                'f1_score': 1.0 - trial.value,
                **trial.params
            }
            
          
            metrics = ['best_f1', 'best_precision', 'best_recall', 'best_misclassified_attacks']
            for metric in metrics:
                if metric in trial.user_attrs:
                    result[metric] = trial.user_attrs[metric]
            
            best_results.append(result)
    
 
    summary_df = pd.DataFrame(best_results)
    summary_csv = f"optuna_summary_{timestamp}.csv"
    summary_df.to_csv(summary_csv, index=False)
    
    print(f"\nDetailed results saved to: {csv_filename}")
    print(f"Summary results saved to: {summary_csv}")
    
    return study, csv_filename, summary_csv

In [ ]:
study, csv_file, summary_file = run_optuna_study(n_trials=50)
    

summary_df = pd.read_csv(summary_file)
print("\nTop 5 trials:")
top_trials = summary_df.sort_values(by='f1_score', ascending=False).head(5)
print(top_trials)

[I 2025-05-21 15:44:23,604] A new study created in memory with name: lstm_autoencoder_study
[I 2025-05-21 15:53:00,912] Trial 0 finished with value: 0.06944953008665933 and parameters: {'architecture_index': 0, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 70, 'dropout_rate': 0.1, 'activation': 'Tanh', 'sequence_length': 10, 'bidirectional': True, 'num_layers': 1, 'optimizer': 'Adam'}. Best is trial 0 with value: 0.06944953008665933.
[I 2025-05-21 16:00:25,964] Trial 1 finished with value: 0.29649904519414383 and parameters: {'architecture_index': 2, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 70, 'dropout_rate': 0.1, 'activation': 'ReLU', 'sequence_length': 10, 'bidirectional': False, 'num_layers': 1, 'optimizer': 'AdamW'}. Best is trial 0 with value: 0.06944953008665933.
[I 2025-05-21 16:10:52,779] Trial 2 finished with value: 0.07216998412504583 and parameters: {'architecture_index': 0, 'batch_size': 32, 'learning_rate': 0.001, 'epochs': 30, 'dropout_rate': 0.1, 'activ

Best trial: 27
Best F1 Score: 0.9507
Best hyperparameters:
    architecture_index: 0
    batch_size: 32
    learning_rate: 0.001
    epochs: 30
    dropout_rate: 0.0
    activation: LeakyReLU
    sequence_length: 5
    bidirectional: True
    num_layers: 1
    optimizer: RMSprop

Detailed results saved to: optuna_trials_20250521_154423.csv
Summary results saved to: optuna_summary_20250521_154423.csv

Top 5 trials:
    trial_number  f1_score  architecture_index  batch_size  learning_rate  \
27            27  0.950688                   0          32          0.001   
30            30  0.949449                   0          32          0.001   
46            46  0.948827                   0          32          0.001   
44            44  0.948721                   0          32          0.001   
32            32  0.948261                   0          32          0.001   

    epochs  dropout_rate activation  sequence_length  bidirectional  \
27      30           0.0  LeakyReLU             